# 02 · Fine-tune FinBERT on the LLM labels, score all headlines, check against our 100 hand labels

**What it does:**
1. Fine-tunes FinBERT (`ProsusAI/finbert`) with 4 classes (bullish / neutral / bearish / not relevant) on the LLM labels from notebook 01, in two versions: **headline only** and **headline + first paragraph**. **Training uses only headlines before `TRAIN_END` (T0 + 1 month = 28 Mar 2026)**, so the later test period stays fully untouched; then all headlines are scored.
2. Scores all ~9,500 headlines → `headlines_finetuned_scores.parquet` for Ninon (same `id`, `news_day` columns as the baseline file, so she just swaps it in).
3. Compares plain FinBERT, the LLM and fine-tuned FinBERT against **our 100 hand labels** (accuracy, macro-F1) and measures how much we agree (Cohen's kappa).

**Needs in `My Drive/capesize_news/`:** `headlines_clean.parquet`, `llm_labels.parquet` (from 01), and for step 3 `check_100_christella.xlsx`, `check_100_ninon.xlsx` (filled in) and `headlines_finbert_baseline.parquet`.
**Runtime:** T4 GPU, about 10–20 min.

**Why this is not leakage:** the labels come only from the text of each headline (no prices, no future), so training on headlines from the whole window does not tell the model anything about later freight rates. The real test, whether the scores improve the 5TC forecast, happens in the team's walk-forward evaluation.

In [ ]:
# 1. Connect Google Drive and point to the data folder
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DATA = Path('/content/drive/MyDrive/capesize_news')   # <- your private Drive folder (do not share it: licensed Reuters text)
for f in ['headlines_clean.parquet']:
    print(f, 'OK' if (DATA / f).exists() else 'MISSING: upload it to ' + str(DATA))

In [ ]:
# 2. Load the merged headlines (from Ninon) and build the text the model sees
import pandas as pd, numpy as np
df = pd.read_parquet(DATA / 'headlines_clean.parquet')
df['lead'] = df['lead'].fillna('').str.strip()
df['has_lead'] = df['lead'].str.len() >= 40          # shorter "leads" are editorial notes or datelines
df['text_head'] = df['title'].fillna('').str.strip()
df['text_headlead'] = np.where(df['has_lead'], df['text_head'] + '. ' + df['lead'].str.slice(0, 300), df['text_head'])
df['period'] = np.where(pd.to_datetime(df['news_day']) < '2026-02-28', 'before closure', 'after closure')
print(len(df), 'headlines |', df['source'].value_counts().to_dict(), '| usable lead:', f"{df['has_lead'].mean():.0%}")

lab = pd.read_parquet(DATA / 'llm_labels.parquet')[['id', 'llm_label', 'p_bullish', 'p_neutral', 'p_bearish', 'p_notrel', 'llm_score']]
df = df.merge(lab, on='id', how='left')
print('LLM labels for', df['llm_label'].notna().sum(), 'headlines')

In [ ]:
# 3. Training set = LLM-labelled headlines, minus the 100 hand-check headlines (kept apart as the test set)
check_ids = set(pd.read_excel(DATA / 'check_100_christella.xlsx')['id']) if (DATA / 'check_100_christella.xlsx').exists() else set()
CLASSES = ['bullish', 'neutral', 'bearish', 'not relevant']
TRAIN_END = pd.Timestamp('2026-03-28')   # T0 (28 Feb 2026, Hormuz closure) + 1 month: train only on headlines before this day
data = df[df['llm_label'].notna() & ~df['id'].isin(check_ids) & (pd.to_datetime(df['news_day']) < TRAIN_END)].copy()
print(f'Training period: up to {TRAIN_END.date()} -> {len(data)} headlines (of {len(df)}); later headlines are only scored, never trained on')
data['y'] = data['llm_label'].map({c: i for i, c in enumerate(CLASSES)})
from sklearn.model_selection import train_test_split
train, val = train_test_split(data, test_size=0.1, random_state=42, stratify=data['y'])
print(f'train {len(train)} | validation {len(val)} | held-out hand-check {len(check_ids)}')
print(train['llm_label'].value_counts(normalize=True).round(3).to_string())

In [ ]:
# 4. Fine-tuning (small, plain PyTorch loop: 3 epochs, keep the epoch with the best validation macro-F1)
import torch, copy
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score
dev = 'cuda'
BASE = 'ProsusAI/finbert'
ftok = AutoTokenizer.from_pretrained(BASE)

def batches(texts, ys=None, bs=32, shuffle=False, max_len=64):
    idx = np.random.RandomState(42).permutation(len(texts)) if shuffle else np.arange(len(texts))
    for i in range(0, len(texts), bs):
        j = idx[i:i + bs]
        enc = ftok([texts[k] for k in j], truncation=True, max_length=max_len, padding=True, return_tensors='pt')
        yield {k: v.to(dev) for k, v in enc.items()}, (torch.tensor(ys[j]).to(dev) if ys is not None else None)

@torch.no_grad()
def predict(model, texts, max_len):
    model.eval(); out = []
    for enc, _ in batches(texts, bs=128, max_len=max_len):
        out.append(torch.softmax(model(**enc).logits.float(), -1).cpu().numpy())
    return np.vstack(out)

def finetune(col, max_len, epochs=3, lr=2e-5):
    torch.manual_seed(42)
    model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=4, ignore_mismatched_sizes=True,
              id2label=dict(enumerate(CLASSES)), label2id={c: i for i, c in enumerate(CLASSES)}).to(dev)
    xt, yt = train[col].tolist(), train['y'].to_numpy()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * int(np.ceil(len(xt) / 32)); sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.cuda.amp.GradScaler(); best, best_f1 = None, -1
    for ep in range(epochs):
        model.train()
        for enc, y in batches(xt, yt, shuffle=True, max_len=max_len):
            with torch.autocast('cuda', dtype=torch.float16):
                loss = model(**enc, labels=y).loss
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
        pv = predict(model, val[col].tolist(), max_len).argmax(1)
        f1 = f1_score(val['y'], pv, average='macro')
        print(f'{col} epoch {ep + 1}: validation accuracy {accuracy_score(val["y"], pv):.3f}, macro-F1 {f1:.3f}')
        if f1 > best_f1: best_f1, best = f1, copy.deepcopy(model.state_dict())
    model.load_state_dict(best); return model

models = {'head': finetune('text_head', 64), 'headlead': finetune('text_headlead', 160)}

In [ ]:
# 5. Score ALL headlines with both fine-tuned versions -> file for Ninon (same id / news_day as the baseline)
res = df[['id', 'news_day', 'source', 'has_lead']].copy()
for name, col, ml in [('head', 'text_head', 64), ('headlead', 'text_headlead', 160)]:
    p = predict(models[name], df[col].tolist(), ml)
    for k, c in enumerate(['bullish', 'neutral', 'bearish', 'notrel']):
        res[f'ft_{name}_p_{c}'] = p[:, k]
    res[f'ft_{name}_score'] = res[f'ft_{name}_p_bullish'] - res[f'ft_{name}_p_bearish']   # -1 bearish ... +1 bullish
    res[f'ft_{name}_label'] = np.array(CLASSES)[p.argmax(1)]
res['llm_score'] = df['llm_score'].to_numpy(); res['llm_label'] = df['llm_label'].to_numpy()
res.to_parquet(DATA / 'headlines_finetuned_scores.parquet', index=False)
print('saved headlines_finetuned_scores.parquet', res.shape)
print(res.groupby(df['period'])[['ft_head_score', 'ft_headlead_score', 'llm_score']].mean().round(3))
for name, m in models.items():                       # keep the models (Drive, private)
    m.save_pretrained(DATA / f'finbert_capesize_{name}'); ftok.save_pretrained(DATA / f'finbert_capesize_{name}')

In [ ]:
# 6. Numbers for the report: label shares (LLM and fine-tuned), overall and before/after the closure -> run_summary.md
shares = pd.DataFrame({
    'LLM (Qwen2.5-7B)': df['llm_label'].value_counts(normalize=True),
    'Fine-tuned FinBERT (headline + lead)': res['ft_headlead_label'].value_counts(normalize=True),
    'Fine-tuned FinBERT (headline)': res['ft_head_label'].value_counts(normalize=True)}).reindex(CLASSES).round(3)
by_period = pd.crosstab(df['period'], df['llm_label'], normalize='index').reindex(columns=CLASSES).round(3)
display(shares); display(by_period)
txt = (f"# Run summary\n\nHeadlines: {len(df)} | LLM: Qwen/Qwen2.5-7B-Instruct (4-bit) | fine-tuned: ProsusAI/finbert, 4 classes, "
       f"3 epochs, lr 2e-5, batch 32, best epoch by validation macro-F1 | training headlines: {len(train)} (+{len(val)} validation), "
       f"news_day < {TRAIN_END.date()}, 100 hand-check headlines excluded\n\n## Label shares\n\n{shares.to_markdown()}\n\n"
       f"## LLM label shares before / after the closure (28 Feb 2026)\n\n{by_period.to_markdown()}\n")
(DATA / 'run_summary.md').write_text(txt); print('saved run_summary.md')

In [ ]:
# 7. How good is each model? Compare with OUR 100 hand labels (run after both check sheets are filled in)
from sklearn.metrics import cohen_kappa_score, confusion_matrix
L = {'A': 'bullish', 'B': 'neutral', 'C': 'bearish', 'D': 'not relevant'}
c = pd.read_excel(DATA / 'check_100_christella.xlsx')[['id', 'label (A/B/C/D)']].rename(columns={'label (A/B/C/D)': 'christella'})
n = pd.read_excel(DATA / 'check_100_ninon.xlsx')[['id', 'label (A/B/C/D)']].rename(columns={'label (A/B/C/D)': 'ninon'})
h = c.merge(n, on='id').dropna()
if len(h) < 20:
    raise SystemExit(f'Only {len(h)} headlines hand-labelled by both so far: fill in both check sheets, then run this cell again.')
h[['christella', 'ninon']] = h[['christella', 'ninon']].apply(lambda s: s.str.strip().str.upper().map(L))
print(f'Hand-labelled by both: {len(h)} | agreement {np.mean(h.christella == h.ninon):.0%} | '
      f"Cohen's kappa {cohen_kappa_score(h.christella, h.ninon):.2f}  (0.4-0.6 moderate, 0.6-0.8 good, >0.8 very good)")
gold = h[h.christella == h.ninon][['id', 'christella']].rename(columns={'christella': 'gold'})
print(f'Gold set = the {len(gold)} headlines where we agree\n')

base = pd.read_parquet(DATA / 'headlines_finbert_baseline.parquet')
def find(word):   # baseline column names differ between versions: prefer headline+lead, else headline
    hits = [c for c in base.columns if word in c.lower()]
    lead = [c for c in hits if 'lead' in c.lower()]
    if not hits: raise KeyError(f'no baseline column with "{word}" in {base.columns.tolist()}')
    return (lead or hits)[0]
pcols = [find('pos'), find('neg'), find('neu')]
print('Plain FinBERT columns used:', pcols)
base['plain_finbert'] = np.array(['bullish', 'bearish', 'neutral'])[base[pcols].to_numpy().argmax(1)]   # positive -> bullish
g = gold.merge(res[['id', 'llm_label', 'ft_head_label', 'ft_headlead_label']], on='id').merge(base[['id', 'plain_finbert']], on='id')
rows = []
for col, name in [('plain_finbert', 'Plain FinBERT (baseline, 3 classes)'), ('llm_label', 'LLM (Qwen2.5-7B)'),
                  ('ft_head_label', 'Fine-tuned FinBERT, headline'), ('ft_headlead_label', 'Fine-tuned FinBERT, headline + lead')]:
    rows.append({'model': name, 'accuracy': accuracy_score(g['gold'], g[col]),
                 'macro-F1': f1_score(g['gold'], g[col], average='macro', labels=CLASSES, zero_division=0)})
display(pd.DataFrame(rows).round(3))
print('Confusion matrix, fine-tuned headline + lead (rows = our label, columns = model):')
display(pd.DataFrame(confusion_matrix(g['gold'], g['ft_headlead_label'], labels=CLASSES), index=CLASSES, columns=CLASSES))